In [ ]:
!pip install earthengine-api pandas --quiet

import pandas as pd
import ee
from google.colab import auth
from google.colab import files
import warnings
warnings.filterwarnings('ignore')

print("🔐 A autenticar os serviços da Google...")
auth.authenticate_user()

print("🌍 A ligar ao Google Earth Engine...")
ee.Initialize(project='spatial-garage-267618')

df_base = pd.read_csv('tcc_matriz_final.csv', sep=';', decimal=',')
lista_municipios = df_base['Municipio'].unique().tolist()

municipios_fc = ee.FeatureCollection("FAO/GAUL/2015/level2") \
                  .filter(ee.Filter.inList('ADM2_NAME', lista_municipios))

mapbiomas = ee.Image('projects/mapbiomas-public/assets/brazil/lulc/collection9/mapbiomas_collection90_integration_v1')

resultados = []
anos = list(range(2010, 2024))

def aplicar_fator_escala(image):
    bandas_opticas = image.select('SR_B.').multiply(0.0000275).add(-0.2)
    return image.addBands(bandas_opticas, None, True)

for ano in anos:
    print(f"🛰️ A extrair o Percentil 10 de Estresse para o ano {ano}...")
    mascara_cafe = mapbiomas.select(f'classification_{ano}').eq(46)

    colecao_sat = 'LANDSAT/LC08/C02/T1_L2' if ano >= 2013 else 'LANDSAT/LE07/C02/T1_L2'
    banda_nir = 'SR_B5' if ano >= 2013 else 'SR_B4'
    banda_red = 'SR_B4' if ano >= 2013 else 'SR_B3'

    def processar_satelite(image):
        qa = image.select('QA_PIXEL')
        mask_nuvem = qa.bitwiseAnd(1 << 3).eq(0).And(qa.bitwiseAnd(1 << 4).eq(0))
        img_calibrada = aplicar_fator_escala(image)
        ndvi = img_calibrada.normalizedDifference([banda_nir, banda_red]).rename('NDVI_Estresse')
        mask_final = mask_nuvem.And(mascara_cafe)
        return img_calibrada.updateMask(mask_final).addBands(ndvi)

    imagens_ano = ee.ImageCollection(colecao_sat) \
                    .filterBounds(municipios_fc) \
                    .filterDate(f'{ano}-01-01', f'{ano}-12-31') \
                    .map(processar_satelite)

    # A MÁGICA: Em vez do mínimo absoluto, pegamos o Percentil 10 (elimina sombras e ruídos!)
    imagem_percentil = imagens_ano.select('NDVI_Estresse').reduce(ee.Reducer.percentile([10])).rename('NDVI_Minimo')

    estatisticas = imagem_percentil.reduceRegions(
        collection=municipios_fc,
        reducer=ee.Reducer.mean(),
        scale=30
    ).getInfo()

    for feature in estatisticas['features']:
        nome_mun = feature['properties'].get('ADM2_NAME')
        valor_ndvi = feature['properties'].get('mean')

        if valor_ndvi is not None:
            resultados.append({'Ano': ano, 'Municipio': nome_mun, 'NDVI_Minimo': valor_ndvi})

df_satelite = pd.DataFrame(resultados)

print("\n🔗 A compilar a tabela final sem ruídos...")
df_fase2_definitiva = pd.merge(df_base, df_satelite, on=['Ano', 'Municipio'], how='inner')

nome_ficheiro = 'TCC_Matriz_Percentil_Perfeita.csv'
df_fase2_definitiva.to_csv(nome_ficheiro, index=False, sep=';', decimal=',')
files.download(nome_ficheiro)

print("✅ Ficheiro gerado com o filtro de Percentil 10 aplicado!")

🔐 A autenticar os serviços da Google...
🌍 A ligar ao Google Earth Engine...
🛰️ A extrair o Percentil 10 de Estresse para o ano 2010...
🛰️ A extrair o Percentil 10 de Estresse para o ano 2011...
🛰️ A extrair o Percentil 10 de Estresse para o ano 2012...
🛰️ A extrair o Percentil 10 de Estresse para o ano 2013...
🛰️ A extrair o Percentil 10 de Estresse para o ano 2014...
🛰️ A extrair o Percentil 10 de Estresse para o ano 2015...
🛰️ A extrair o Percentil 10 de Estresse para o ano 2016...
🛰️ A extrair o Percentil 10 de Estresse para o ano 2017...
🛰️ A extrair o Percentil 10 de Estresse para o ano 2018...
🛰️ A extrair o Percentil 10 de Estresse para o ano 2019...
🛰️ A extrair o Percentil 10 de Estresse para o ano 2020...
🛰️ A extrair o Percentil 10 de Estresse para o ano 2021...
🛰️ A extrair o Percentil 10 de Estresse para o ano 2022...
🛰️ A extrair o Percentil 10 de Estresse para o ano 2023...

🔗 A compilar a tabela final sem ruídos...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Ficheiro gerado com o filtro de Percentil 10 aplicado!


In [ ]:
!pip install earthengine-api pandas --quiet

import pandas as pd
import ee
from google.colab import auth
from google.colab import files
import warnings
warnings.filterwarnings('ignore')

print("🔐 A autenticar os serviços da Google...")
auth.authenticate_user()

print("🌍 A ligar ao Google Earth Engine...")
# ATENÇÃO: Coloque o ID do seu projeto do GEE aqui dentro das aspas!
ee.Initialize(project='spatial-garage-267618')

print("📊 A ler a matriz base de dados do IBGE e do Clima...")
# Partimos do ficheiro original seguro
df_base = pd.read_csv('tcc_matriz_final.csv', sep=';', decimal=',')
lista_municipios = df_base['Municipio'].unique().tolist()

municipios_fc = ee.FeatureCollection("FAO/GAUL/2015/level2") \
                  .filter(ee.Filter.inList('ADM2_NAME', lista_municipios))

mapbiomas = ee.Image('projects/mapbiomas-public/assets/brazil/lulc/collection9/mapbiomas_collection90_integration_v1')

resultados_fenologicos = []
anos = list(range(2010, 2024))

# Função de calibração óptica oficial da NASA para o Landsat Coleção 2
def calibrar_e_calcular_ndvi(image, mascara_cafe):
    # Máscara de Nuvem e Sombra
    qa = image.select('QA_PIXEL')
    mask_nuvem = qa.bitwiseAnd(1 << 3).eq(0).And(qa.bitwiseAnd(1 << 4).eq(0))

    # Aplicar Fator de Escala
    bandas_opticas = image.select('SR_B.').multiply(0.0000275).add(-0.2)
    img_calibrada = image.addBands(bandas_opticas, None, True)

    # Calcular NDVI de acordo com o satélite
    sat_id = image.get('SPACECRAFT_ID')
    banda_nir = ee.Algorithms.If(ee.String(sat_id).equals('LANDSAT_8'), 'SR_B5', 'SR_B4')
    banda_red = ee.Algorithms.If(ee.String(sat_id).equals('LANDSAT_8'), 'SR_B4', 'SR_B3')

    ndvi = img_calibrada.normalizedDifference([banda_nir, banda_red]).rename('NDVI')

    # O pixel tem de ser Café E estar sem nuvens
    mask_final = mask_nuvem.And(mascara_cafe)
    return img_calibrada.updateMask(mask_final).addBands(ndvi)

for ano in anos:
    print(f"🛰️ A processar a fenologia completa para o ano {ano}...")
    mascara_cafe = mapbiomas.select(f'classification_{ano}').eq(46)
    colecao_sat = 'LANDSAT/LC08/C02/T1_L2' if ano >= 2013 else 'LANDSAT/LE07/C02/T1_L2'

    # 1. Coleção do Ano Inteiro
    imagens_ano = ee.ImageCollection(colecao_sat) \
                    .filterBounds(municipios_fc) \
                    .filterDate(f'{ano}-01-01', f'{ano}-12-31') \
                    .map(lambda img: calibrar_e_calcular_ndvi(img, mascara_cafe))

    # 2. Coleção restrita à Primavera (Florada: Setembro a Novembro)
    imagens_florada = imagens_ano.filterDate(f'{ano}-09-01', f'{ano}-11-30')

    # 3. Extração dos Redutores Fenológicos (Ignorando ruídos extremos com Percentis 10 e 90)
    ndvi_p10 = imagens_ano.select('NDVI').reduce(ee.Reducer.percentile([10])).rename('NDVI_P10_Fundo')
    ndvi_p90 = imagens_ano.select('NDVI').reduce(ee.Reducer.percentile([90])).rename('NDVI_P90_Teto')
    ndvi_florada = imagens_florada.select('NDVI').mean().rename('NDVI_Media_Florada')

    # Juntar todas as métricas numa única imagem multicamadas
    imagem_fenologica = ee.Image.cat([ndvi_p10, ndvi_p90, ndvi_florada])

    # Calcular a média destas métricas para cada polígono de município
    estatisticas = imagem_fenologica.reduceRegions(
        collection=municipios_fc,
        reducer=ee.Reducer.mean(),
        scale=30
    ).getInfo()

    for feature in estatisticas['features']:
        nome_mun = feature['properties'].get('ADM2_NAME')
        v_p10 = feature['properties'].get('NDVI_P10_Fundo')
        v_p90 = feature['properties'].get('NDVI_P90_Teto')
        v_florada = feature['properties'].get('NDVI_Media_Florada')

        if v_p10 is not None and v_p90 is not None:
            resultados_fenologicos.append({
                'Ano': ano,
                'Municipio': nome_mun,
                'NDVI_Vigor_Maximo': v_p90,
                'NDVI_Estresse_Maximo': v_p10,
                'NDVI_Florada': v_florada
            })

df_satelite = pd.DataFrame(resultados_fenologicos)

# Calcular a Amplitude (O tamanho da perda de folhas)
df_satelite['NDVI_Amplitude'] = df_satelite['NDVI_Vigor_Maximo'] - df_satelite['NDVI_Estresse_Maximo']

print("\n🔗 A fundir os dados espaciais com a matriz agronómica...")
# O merge 'inner' garante que os municípios sem dados de satélite validados são descartados
df_matriz_suprema = pd.merge(df_base, df_satelite, on=['Ano', 'Municipio'], how='inner')

nome_ficheiro = 'TCC_Matriz_Fenologica_SUPREMA.csv'
df_matriz_suprema.to_csv(nome_ficheiro, index=False, sep=';', decimal=',')
files.download(nome_ficheiro)

print(f"✅ Extração de Elite concluída! O ficheiro '{nome_ficheiro}' está a ser transferido.")

🔐 A autenticar os serviços da Google...
🌍 A ligar ao Google Earth Engine...
📊 A ler a matriz base de dados do IBGE e do Clima...
🛰️ A processar a fenologia completa para o ano 2010...
🛰️ A processar a fenologia completa para o ano 2011...
🛰️ A processar a fenologia completa para o ano 2012...
🛰️ A processar a fenologia completa para o ano 2013...
🛰️ A processar a fenologia completa para o ano 2014...
🛰️ A processar a fenologia completa para o ano 2015...
🛰️ A processar a fenologia completa para o ano 2016...
🛰️ A processar a fenologia completa para o ano 2017...
🛰️ A processar a fenologia completa para o ano 2018...
🛰️ A processar a fenologia completa para o ano 2019...
🛰️ A processar a fenologia completa para o ano 2020...
🛰️ A processar a fenologia completa para o ano 2021...
🛰️ A processar a fenologia completa para o ano 2022...
🛰️ A processar a fenologia completa para o ano 2023...

🔗 A fundir os dados espaciais com a matriz agronómica...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Extração de Elite concluída! O ficheiro 'TCC_Matriz_Fenologica_SUPREMA.csv' está a ser transferido.
